# Phase 3 — Baseline 3 (memory-safe): Multi-Stream LSTM

**Why this version exists:** the original notebook caused kernel crashes. This
rewrite eliminates the most likely causes:

1. **No autoreload** — autoreload can corrupt module state and crash kernels on long runs.
2. **Bundle loaded on demand**, one year-pair at a time (4 train years + 1 test year),
   instead of holding all 5 years in RAM simultaneously alongside concatenated train sets.
3. **Mask stays bool**, not float — saves ~700 MB on the train set.
4. **Per-fold cleanup**: model, optimizer, dataloaders, and CUDA cache are all freed
   between folds. GC is forced.
5. **Per-fold checkpoint** of GPU memory + system RAM so we can see if anything leaks.
6. **Smaller default hidden** (32, not 64) — halves param count and activations.
7. **Mixed precision training** when CUDA is available — halves activation memory.
8. **Smaller batch size default** (32, not 64) — halves peak activation memory.

The trade-off: training is somewhat slower per epoch, but it should not crash.


## 1. Setup

In [2]:
from pathlib import Path
import sys, os, time, gc, psutil
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'Total GPU memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def memstat(tag=''):
    cpu = psutil.Process(os.getpid()).memory_info().rss / 1e9
    if torch.cuda.is_available():
        gpu_alloc = torch.cuda.memory_allocated() / 1e9
        gpu_reserved = torch.cuda.memory_reserved() / 1e9
        return f'{tag} RAM={cpu:.2f}GB  GPU_alloc={gpu_alloc:.2f}GB  GPU_reserved={gpu_reserved:.2f}GB'
    return f'{tag} RAM={cpu:.2f}GB'

print(memstat('init:'))


torch: 2.7.1+cu118
CUDA available: True
Device: NVIDIA A100-PCIE-40GB MIG 1g.5gb
Total GPU memory: 5.10 GB
init: RAM=0.44GB  GPU_alloc=0.00GB  GPU_reserved=0.00GB


In [3]:
TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'
OUTPUT_DIR  = PROJECT_ROOT / 'phase3_outputs' / 'lstm'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === KNOBS (smaller defaults for memory safety) ===
YEARS       = [2017, 2018, 2019, 2020, 2021]
STREAMS     = ['S1_only', 'S2_only', 'Met_only', 'S1+S2', 'All']
POLICIES    = ['all10', 'rank1']
EPOCHS      = 15
HIDDEN      = 32       # was 64
BATCH_SIZE  = 32       # was 64
LR          = 1e-3
USE_AMP     = True     # automatic mixed precision when CUDA

print(f'epochs={EPOCHS} hidden={HIDDEN} batch={BATCH_SIZE} amp={USE_AMP}')


epochs=15 hidden=32 batch=32 amp=True


## 2. Helper: per-fold loader that doesn't keep concatenated sets in RAM

In [9]:
from phase3_baselines.dataset import load_year_bundle, SelectionPolicy, apply_selection, concat_bundles
from phase3_baselines.baseline_lstm import (MultiStreamLSTM, soft_label_loss,
                                              STREAM_CONFIGS, stream_indices)
from phase3_baselines.evaluate import evaluate_predictions
from phase3_baselines.resumable import append_row, already_done, save_pred_npz


def load_loyo_for_year(test_year: int, policy: SelectionPolicy):
    """Load 4 train years + 1 test year. Releases memory aggressively."""
    test_b = apply_selection(load_year_bundle(TENSORS_DIR / f'phenology_{test_year}.npz'), policy)
    train_parts = []
    for y in YEARS:
        if y == test_year: continue
        b = apply_selection(load_year_bundle(TENSORS_DIR / f'phenology_{y}.npz'), policy)
        train_parts.append(b)
    # We have to concat for training (one DataLoader); but we can do it lazily.
    # For memory: convert each part to a dict, concat all-at-once, drop the parts.
    train_b = concat_bundles({y: b for y, b in zip([y for y in YEARS if y != test_year], train_parts)},
                              years=[y for y in YEARS if y != test_year])
    del train_parts
    gc.collect()
    return train_b, test_b


## 3. Memory-safe training function

In [6]:
class TensorBundle:
    """Holds tensors in CPU pinned memory; index into them on the fly to avoid copies."""
    def __init__(self, bundle):
        # Keep mask as bool to save memory; convert to float in the forward pass
        self.X    = torch.from_numpy(bundle['X'])            # float32
        self.M    = torch.from_numpy(bundle['mask'])         # bool
        self.Y    = torch.from_numpy(bundle['Y'])            # float32
        self.LM   = torch.from_numpy(bundle['label_mask'])   # bool
        self.bag  = torch.from_numpy(bundle['bag_id'])
        self.rank = torch.from_numpy(bundle['rank'])
        self.yr   = torch.from_numpy(bundle['harvest_year'])
        self.xid  = torch.from_numpy(bundle['xid'])
        self.N    = self.X.shape[0]
    def __len__(self): return self.N


def iter_batches(tb: TensorBundle, batch_size: int, shuffle: bool, device, generator=None):
    idx = torch.randperm(tb.N, generator=generator) if shuffle else torch.arange(tb.N)
    for i in range(0, tb.N, batch_size):
        sl = idx[i:i+batch_size]
        # Move only this batch to GPU (avoids materializing whole-bundle tensors on device)
        x = tb.X[sl].to(device, non_blocking=True)
        m = tb.M[sl].to(device, non_blocking=True).to(torch.float32)
        y = tb.Y[sl].to(device, non_blocking=True)
        lm = tb.LM[sl].to(device, non_blocking=True)
        yield x, m, y, lm, sl


def train_one_fold_safe(train_b, test_b, active, hidden=32, epochs=10,
                          batch_size=32, lr=1e-3, weight_decay=1e-5,
                          use_amp=True, device=DEVICE, verbose=False):
    """Train one fold with explicit per-batch GPU streaming + AMP."""
    tb_tr = TensorBundle(train_b)
    tb_te = TensorBundle(test_b)
    n_phases = train_b['Y'].shape[2]
    feature_names = train_b['feature_names']

    model = MultiStreamLSTM(feature_names=feature_names, hidden=hidden,
                             num_layers=1, n_phases=n_phases, dropout=0.1).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler = torch.amp.GradScaler('cuda', enabled=(use_amp and device.type=='cuda'))
    rng = torch.Generator().manual_seed(0)

    for ep in range(epochs):
        model.train()
        total, nb = 0.0, 0
        for x, m, y, lm, _ in iter_batches(tb_tr, batch_size, True, device, rng):
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(use_amp and device.type=='cuda')):
                logits = model(x, m, active=active)
                loss = soft_label_loss(logits, y, lm)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            total += loss.item(); nb += 1
        if verbose:
            print(f'    epoch {ep+1:2d}: loss={total/max(nb,1):.4f}')

    # Predict
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for x, m, y, lm, _ in iter_batches(tb_te, batch_size, False, device):
            with torch.amp.autocast('cuda', enabled=(use_amp and device.type=='cuda')):
                logits = model(x, m, active=active)
            pred_day = logits.argmax(dim=1).cpu().numpy()
            truth_day = y.argmax(dim=1).cpu().numpy()
            truth_day[~lm.cpu().numpy()] = -1
            preds.append(pred_day); truths.append(truth_day)
    pred_all  = np.concatenate(preds)
    truth_all = np.concatenate(truths)
    xid  = tb_te.xid.numpy()
    bag  = tb_te.bag.numpy()
    rank_ = tb_te.rank.numpy()
    yr   = tb_te.yr.numpy()

    # Cleanup
    del model, opt, scaler, tb_tr, tb_te
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return pred_all, truth_all, xid, bag, rank_, yr


## 4. Smoke test (one fold, 3 epochs)

In [7]:
print(memstat('before smoke:'))
policy = SelectionPolicy(use_only_rank1=True)
train_b, test_b = load_loyo_for_year(2020, policy)
print(f'train N={len(train_b["xid"])}, test N={len(test_b["xid"])}')
print(memstat('after load:'))

t0 = time.time()
pred, truth, xid, bag, rank, yr = train_one_fold_safe(
    train_b, test_b, active=STREAM_CONFIGS['All'],
    hidden=32, epochs=10, batch_size=BATCH_SIZE, lr=LR, use_amp=USE_AMP,
    verbose=True,
)
print(f'\nSmoke fold completed in {time.time()-t0:.1f}s')
print(memstat('after smoke:'))

phase_ids = train_b['phase_ids']
del train_b, test_b
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

from phase3_baselines.evaluate import evaluate_predictions, print_evaluation
report = evaluate_predictions(pred_days=pred.astype(np.float32),
                                true_days=truth.astype(np.float32),
                                phase_ids=phase_ids, years=yr, ranks=rank)
print_evaluation(report, 'Smoke test')
print(memstat('after cleanup:'))


before smoke: RAM=0.44GB  GPU_alloc=0.00GB  GPU_reserved=0.00GB
train N=2421, test N=585
after load: RAM=0.60GB  GPU_alloc=0.00GB  GPU_reserved=0.00GB
    epoch  1: loss=0.2494
    epoch  2: loss=0.1179
    epoch  3: loss=0.1023
    epoch  4: loss=0.0877
    epoch  5: loss=0.0771
    epoch  6: loss=0.0724
    epoch  7: loss=0.0700
    epoch  8: loss=0.0682
    epoch  9: loss=0.0667
    epoch 10: loss=0.0658

Smoke fold completed in 20.7s
after smoke: RAM=1.12GB  GPU_alloc=0.02GB  GPU_reserved=0.05GB

Smoke test

[Headline — macro over phases]
  macro_mae               : 8.957
  macro_r2                : -0.390
  macro_within_5d         : 0.418
  n_phases_evaluated      : 7

[Per phase]
         n     mae    rmse     r2  within_5d   bias
phase                                              
10     528  11.591  14.765 -0.477      0.335 -5.053
12     518  10.475  13.577 -0.030      0.369 -2.089
15     489  10.536  17.716 -1.731      0.333 -0.716
18     537   7.525   9.800 -0.356      0.479 

## 5. Full grid (resumable, one fold at a time, aggressive cleanup)

In [7]:
# In a new cell, replace the smoke test with:
policy = SelectionPolicy(use_only_rank1=True)
train_b, test_b = load_loyo_for_year(2020, policy)

t0 = time.time()
pred, truth, xid, bag, rank, yr = train_one_fold_safe(
    train_b, test_b, active=STREAM_CONFIGS['All'],
    hidden=32,          # MATCH the full-run config
    epochs=10,          # MATCH the full-run config
    batch_size=BATCH_SIZE, lr=LR, use_amp=USE_AMP,
    verbose=True,        # show all 10 epochs' losses
)
print(f'\nFold completed in {time.time()-t0:.1f}s')
print(memstat('after deep smoke:'))

phase_ids = train_b['phase_ids']
del train_b, test_b
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

from phase3_baselines.evaluate import evaluate_predictions, print_evaluation
report = evaluate_predictions(
    pred_days=pred.astype(np.float32), true_days=truth.astype(np.float32),
    phase_ids=phase_ids, years=yr, ranks=rank
)
print_evaluation(report, 'Deep smoke (hidden=32, epochs=10)')

    epoch  1: loss=0.2309
    epoch  2: loss=0.1157
    epoch  3: loss=0.0979
    epoch  4: loss=0.0828
    epoch  5: loss=0.0762
    epoch  6: loss=0.0725
    epoch  7: loss=0.0703
    epoch  8: loss=0.0683
    epoch  9: loss=0.0667
    epoch 10: loss=0.0660

Fold completed in 16.6s
after deep smoke: RAM=1.21GB  GPU_alloc=0.02GB  GPU_reserved=0.04GB

Deep smoke (hidden=32, epochs=10)

[Headline — macro over phases]
  macro_mae               : 9.205
  macro_r2                : -0.286
  macro_within_5d         : 0.399
  n_phases_evaluated      : 7

[Per phase]
         n     mae    rmse     r2  within_5d   bias
phase                                              
10     528  11.379  13.943 -0.317      0.303 -5.739
12     518  11.278  14.321 -0.145      0.311 -4.641
15     489  10.980  14.613 -0.858      0.321  2.284
18     537   7.285   9.739 -0.339      0.508  2.331
19     471   9.503  12.061 -0.321      0.352  1.316
21     500   7.542   9.936  0.021      0.472 -0.246
24     563   6.471

In [11]:
RESULTS_CSV = OUTPUT_DIR / 'lstm_results.csv'
PREDS_NPZ   = OUTPUT_DIR / 'lstm_predictions.npz'

policy_map = {
    'all10': SelectionPolicy(use_only_rank1=False),
    'rank1': SelectionPolicy(use_only_rank1=True),
}
done = already_done(RESULTS_CSV, ['stream', 'policy', 'test_year'])
print(f'Already-done folds: {len(done)} (of {len(STREAMS)*len(POLICIES)*len(YEARS)})')

total = len(STREAMS) * len(POLICIES) * len(YEARS)
pbar = tqdm(total=total, desc='LSTM folds')

# Cache phase_ids from one quick load
_b0 = load_year_bundle(TENSORS_DIR / f'phenology_{YEARS[0]}.npz')
phase_ids = _b0['phase_ids']
del _b0
gc.collect()

for s in STREAMS:
    active = STREAM_CONFIGS[s]
    for pol_name in POLICIES:
        policy = policy_map[pol_name]
        for test_y in YEARS:
            key = (s, pol_name, int(test_y))
            if key in done:
                pbar.update(1); continue

            t0 = time.time()
            train_b, test_b = load_loyo_for_year(test_y, policy)
            if len(train_b['xid']) == 0 or len(test_b['xid']) == 0:
                del train_b, test_b
                gc.collect()
                pbar.update(1); continue

            try:
                pred, truth, xid_, bag_, rank_, yr_ = train_one_fold_safe(
                    train_b, test_b, active,
                    hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH_SIZE,
                    lr=LR, use_amp=USE_AMP, verbose=False,
                )
            except torch.cuda.OutOfMemoryError as e:
                print(f'  [OOM] {s}/{pol_name}/{test_y}: {e}')
                del train_b, test_b
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                pbar.update(1); continue

            del train_b, test_b
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

            report = evaluate_predictions(
                pred_days=pred.astype(np.float32),
                true_days=truth.astype(np.float32),
                phase_ids=phase_ids, years=yr_, ranks=rank_,
            )
            dt = time.time() - t0

            for phase, m in report['per_phase'].iterrows():
                append_row(RESULTS_CSV, {
                    'stream': s, 'policy': pol_name, 'test_year': int(test_y),
                    'phase': int(phase), 'fit_secs': round(dt, 2),
                    **m.to_dict()
                })
            save_pred_npz(PREDS_NPZ, f'{s}__{pol_name}__{test_y}',
                          xid=xid_, bag=bag_, rank=rank_, year=yr_,
                          yt=truth.astype(np.int32), yp=pred.astype(np.int32))
            macro_mae = report['headline']['macro_mae']
            pbar.set_postfix({'fold': f'{s[:6]}/{pol_name}/{test_y}',
                              'mae': f'{macro_mae:.2f}',
                              'secs': f'{dt:.0f}',
                              'ram_gb': f'{psutil.Process(os.getpid()).memory_info().rss/1e9:.1f}'})
            pbar.update(1)

pbar.close()
print('\nAll folds processed.')
print(memstat('end:'))


Already-done folds: 3 (of 50)


LSTM folds:   0%|          | 0/50 [00:00<?, ?it/s]


All folds processed.
end: RAM=0.98GB  GPU_alloc=0.02GB  GPU_reserved=0.03GB


## 6. Summarize

In [12]:
results = pd.read_csv(RESULTS_CSV)
print(f'Rows: {len(results)}')

headline = (results.groupby(['stream', 'policy'])
                   .agg(macro_mae=('mae', 'mean'),
                        macro_r2 =('r2',  'mean'),
                        macro_within_5d=('within_5d', 'mean'),
                        macro_rmse=('rmse', 'mean'))
                   .reset_index())
headline.to_csv(OUTPUT_DIR / 'lstm_headline.csv', index=False)
print('\n=== Headline (macro over phases, mean across LOYO folds) ===')
print(headline.round(3).to_string(index=False))


Rows: 350

=== Headline (macro over phases, mean across LOYO folds) ===
  stream policy  macro_mae  macro_r2  macro_within_5d  macro_rmse
     All  all10      8.067    -0.133            0.447      10.549
     All  rank1      8.805    -0.346            0.403      11.599
Met_only  all10      9.099    -0.422            0.394      11.722
Met_only  rank1     10.729    -3.470            0.362      17.548
   S1+S2  all10      7.885    -0.015            0.454      10.095
   S1+S2  rank1      8.731    -0.519            0.407      11.815
 S1_only  all10      8.511    -0.262            0.424      11.117
 S1_only  rank1      9.735    -1.342            0.373      15.018
 S2_only  all10      8.032    -0.095            0.438      10.469
 S2_only  rank1      9.776    -1.606            0.381      15.325


In [13]:
table = (results.groupby(['stream', 'phase'])
                .agg(mae=('mae','mean'), r2=('r2','mean'),
                     within_5d=('within_5d','mean'))
                .reset_index())
print('\n=== MAE by (stream, phase) ===')
print(table.pivot(index='phase', columns='stream', values='mae').round(2).to_string())
print('\n=== R² by (stream, phase) ===')
print(table.pivot(index='phase', columns='stream', values='r2').round(2).to_string())
print('\n=== within±5d by (stream, phase) ===')
print(table.pivot(index='phase', columns='stream', values='within_5d').round(2).to_string())



=== MAE by (stream, phase) ===
stream    All  Met_only  S1+S2  S1_only  S2_only
phase                                           
10       9.83     10.50   9.50    10.71     9.61
12      10.67     11.11  10.29    11.90    11.47
15       9.78     11.31  10.07    10.51    10.45
18       6.38      6.98   6.63     7.27     7.10
19       8.06     11.22   8.03     8.51     8.80
21       7.55      9.05   7.13     7.79     7.66
24       6.77      9.23   6.51     7.17     7.23

=== R² by (stream, phase) ===
stream   All  Met_only  S1+S2  S1_only  S2_only
phase                                          
10     -0.37     -0.43  -0.34    -1.02    -0.35
12     -0.17     -0.28  -0.08    -1.31    -1.18
15     -0.15     -1.34  -0.42    -0.37    -0.60
18     -0.37     -2.56  -1.04    -1.13    -1.11
19     -0.15     -5.13  -0.07    -0.47    -0.58
21     -0.41     -1.88   0.03    -0.71    -0.82
24     -0.05     -2.01   0.06    -0.60    -1.32

=== within±5d by (stream, phase) ===
stream   All  Met_only  S1

In [14]:
pol_compare = (results.groupby(['stream','policy'])['mae'].mean()
                       .unstack('policy'))
pol_compare['Δ (all10 − rank1)'] = pol_compare['all10'] - pol_compare['rank1']
print('\n=== Policy comparison: macro MAE ===')
print(pol_compare.round(2).to_string())



=== Policy comparison: macro MAE ===
policy    all10  rank1  Δ (all10 − rank1)
stream                                   
All        8.07   8.81              -0.74
Met_only   9.10  10.73              -1.63
S1+S2      7.88   8.73              -0.85
S1_only    8.51   9.73              -1.22
S2_only    8.03   9.78              -1.74


In [10]:
"""
F8 add-on: save per-day LSTM logits for one fold (All / shared-label / 2020).

The original Phase 3 LSTM saved only argmax predictions, not per-day logits.
F8 needs the full (N, T, P) per-day probability surfaces to draw probability
curves. This add-on retrains the All-stream / shared-label LSTM for the 2020
LOYO fold and saves the per-day logits before argmax.

Paste this as a new cell in any Jupyter session with access to the
`phase3_baselines` package on your HPC. Total runtime: roughly 15 to 25
minutes on the MIG A100 slice (one fold of LSTM training plus inference).

After it finishes, re-run the figure notebook (notebook 09) and F8 will
be generated.
"""
from pathlib import Path
import sys, gc, time
import numpy as np
import torch

PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from phase3_baselines.dataset import (load_year_bundle, SelectionPolicy,
                                       apply_selection, concat_bundles)
from phase3_baselines.baseline_lstm.train import train_lstm_fold
from phase3_baselines.baseline_lstm.model import MultiStreamLSTM   # adjust if your import path differs

TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'
LSTM_DIR = PROJECT_ROOT / 'phase3_outputs' / 'lstm'
LSTM_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

YEARS = [2017, 2018, 2019, 2020, 2021]
TEST_YEAR = 2020
EPOCHS = 15
HIDDEN = 32
LR = 1e-3

# --- Load data ---
print('Loading bundles...')
policy = SelectionPolicy(use_only_rank1=False)
test_b = apply_selection(load_year_bundle(TENSORS_DIR / f'phenology_{TEST_YEAR}.npz'), policy)
train_parts = {}
for y in YEARS:
    if y == TEST_YEAR:
        continue
    train_parts[y] = apply_selection(load_year_bundle(TENSORS_DIR / f'phenology_{y}.npz'), policy)
train_b = concat_bundles(train_parts, years=[y for y in YEARS if y != TEST_YEAR])
del train_parts; gc.collect()
print(f'  test: {test_b["X"].shape}, train: {train_b["X"].shape}')

# --- Train the All-streams LSTM ---
# NOTE: your project's actual training function may have different argument names.
# If train_lstm_fold returns argmax-only predictions, we'll need to add a forward
# pass below to compute logits.
print('Training LSTM (All streams, shared-label policy, 2020 fold)...')
t0 = time.time()
result = train_lstm_fold(
    train_bundle=train_b, test_bundle=test_b,
    active_streams=('S1', 'S2', 'Met'),
    hidden=HIDDEN, epochs=EPOCHS, batch_size=32, lr=LR,
    use_amp=True, device=DEVICE, verbose=True,
    return_model=True,   # IMPORTANT: we need the model to get logits
)
print(f'  done in {time.time()-t0:.0f}s')

# `result` should contain at minimum: the trained model, and we run inference below.
model = result.get('model', None)
if model is None:
    raise RuntimeError(
        'train_lstm_fold did not return the trained model. '
        'Edit train_lstm_fold to return `model` in its result dict, '
        'or rerun this cell with a wrapper that exposes the model.'
    )

# --- Inference with logit output ---
print('Computing per-day logits on the held-out test fold...')
model.eval()

# Build a simple batch iterator matching how the model expects input
from phase3_baselines.lstm.train import _iter_batches  # adjust if your private name differs
all_logits = []
all_yt = []
all_xid = []
all_bag = []

with torch.no_grad():
    for batch in _iter_batches(test_b, batch_size=64, shuffle=False, device=DEVICE):
        # batch must include x (B, T, F), mask (B, T, F), and reference info
        # Forward pass returning logits of shape (B, T, P)
        logits = model(batch['x'], batch['mask'], active=('S1', 'S2', 'Met'))
        all_logits.append(logits.cpu().numpy())
        all_yt.append(batch['y_days'].cpu().numpy())          # (B, P) reference day indices
        all_xid.append(batch['xid'].cpu().numpy())
        all_bag.append(batch['bag_id'].cpu().numpy())

logits_arr = np.concatenate(all_logits, axis=0).astype(np.float32)
yt_arr     = np.concatenate(all_yt, axis=0).astype(np.int32)
xid_arr    = np.concatenate(all_xid, axis=0).astype(np.int64)
bag_arr    = np.concatenate(all_bag, axis=0).astype(np.int64)

print(f'  logits shape: {logits_arr.shape}')   # expect (N, 426, 7)
print(f'  yt     shape: {yt_arr.shape}')

# --- Save to npz keyed for fig8 ---
NPZ = LSTM_DIR / 'lstm_logits.npz'
pref = f'All__all10__{TEST_YEAR}'
arrays = {
    f'{pref}__logits': logits_arr,
    f'{pref}__yt':     yt_arr,
    f'{pref}__xid':    xid_arr,
    f'{pref}__bag':    bag_arr,
}

# If the file already exists, merge
existing = {}
if NPZ.exists():
    with np.load(NPZ, allow_pickle=False) as zz:
        existing = {k: zz[k] for k in zz.files}
    print(f'Merging with {len(existing)} existing arrays')

existing.update(arrays)
np.savez_compressed(NPZ, **existing)
print(f'Saved -> {NPZ}')
print('\nNow re-run notebook 09 and F8 will be produced.')


ModuleNotFoundError: No module named 'phase3_baselines.baseline_lstm.train'; 'phase3_baselines.baseline_lstm' is not a package

---\n**Done.**